# Stage 2 Notebook 58 - Exp2CCC Anchor + cls_separate_path + topk_fixed K=3 + VFL

**The untested config knob.** Across 26 experiments we never enabled `cls_separate_path=True` on `CLRKDLaneHead`. This flag builds a PARALLEL cls-aggregator pathway: separate `scale_blocks_cls`, `scale_fusion_cls`, `fc_cls`, `cross_attn_cls`. Cls features flow through their own ROI gather + cross-attention, disjoint from geometry features.

Hypothesis: the anchor head's cls collapse (gap ≤ 0.015 across 11 anchor-head experiments) is caused by cls and geometry sharing the same per_prior_features. With cls_separate_path=True, cls has its own parameter budget to learn discriminative features that the geometry losses don't pull toward 'average lane-y curve' representations.

Combined with K=3 topk_fixed (NB55's geometry winner) and VFL.

Single new config knob vs NB55 (exp50): `cls_separate_path: false -> true`. Adds ~5% params to the lane head but no other change.

### Run mode
1. `DEBUG_MODE=True` smoke.
2. `DEBUG_MODE=False` 20 ep limit=3000.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint_smoke.log
OK exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.4251 det_loss=3.3470 grad_cos=0.0188 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5006693005561829, 'gate/lane_mean': 0.4989277422428131, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp53_rmt_gca_anchor_cls_sep_topk3_vfl_jo

0

## What to watch in Exp2CCC

Reference NB55 (K=3 topk_fixed, cls_separate_path=False): matched_iou=0.381, gap=0.002, decoded_f1=0.053, val_lane_best_f1=0.139.

Pass criteria at epoch 20:
- `pos_score - neg_score >= 0.03` -- the smoking gun. If separating cls features cracks the anchor head's cls equilibrium, gap should be 10x NB55's.
- `val/matched_line_iou >= 0.40` -- preserve NB55's geometry.
- `val/lane/decoded_f1 >= 0.07` -- 1.3x NB55.
- `val/lane_best_f1 >= 0.20` -- 1.4x NB55.

If gap < 0.01: cls_separate_path doesn't help and the anchor head's cls is fundamentally bottlenecked by the per-prior ROI feature design itself. Confirms we need to abandon the anchor head for cls and use the query head with DAB+DN.

If gap >= 0.05: we have an anchor head that finally does cls. Combine with NB48 full-data geometry in a future Exp2FFF.